In [1]:
## 2025.09.01 상 결합에 대한 논문 작성느라 만듦, 25.12.20 수정
#%% 라이브러리 불러오기
from pathlib import Path
dir_path = Path(r"C:\Users\yu2hy\OneDrive\◎2020_copus\가공_결과물\바른형태분석_말뭉치")
v_folder = dir_path / "최신버전" / "v"
sen_folder = dir_path / "최신버전" / "sen"
save_folder = '2025.09.01_상'

from datetime import datetime
time = []#걸린 시간 체크 리스트
start_t = datetime.now() #시간 체크
time.append(datetime.now() - start_t) #시간간격 저장
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M") #시작 시간 저장

#%% 파일 읽어오기 1분 21초
import pandas as pd
for file in v_folder.glob('*.csv'):
    print(file)
### 1.1 v 파일 읽기: 55초 걸림.
df_v_list = []
for file in v_folder.glob("*.csv"):
    with open(file, "r", encoding = "UTF-8") as f:
        df = pd.read_csv(f, low_memory=False)
        #'Unnamed:'를 포함하는 columns 삭제
        cols_to_drop = [col for col in df.columns if 'Unnamed:' in col]
        df = df.drop(columns=cols_to_drop)
        
        df_v_list.append(df)

df_v_list[2].rename(columns={'docu_id': 'file_id'}, inplace=True)

### 1.2 문장 파일 읽어오기
print('%%%%%1.2 문장 파일-sen2 읽기') 
df_sen_list = []
for file in sen_folder.glob('*.csv'):
    print(file)
    
for file in sen_folder.glob("*.csv"):
    with open(file, "r", encoding = "UTF-8") as f:
        df = pd.read_csv(f, low_memory=False)
        #'Unnamed:'를 포함하는 columns 삭제
        cols_to_drop = [col for col in df.columns if 'Unnamed:' in col]
        df = df.drop(columns=cols_to_drop)
        df_sen_list.append(df)
df_sen_list[2].rename(columns={'docu_id': 'file_id'}, inplace=True)

for i, df in enumerate(df_v_list):
    df['copus'] = i
    df.loc[df['V_No'] == 3035, 'V_No'] = 3025 #갖다, 가지다를 하나로 합함.
    print(f"{i}, {len(df[df['V_label']=="VX"])}")

C:\Users\yu2hy\OneDrive\◎2020_copus\가공_결과물\바른형태분석_말뭉치\최신버전\v\국립국어원 구어 말뭉치(버전 1.2)_일부(1,15)_v5.csv
C:\Users\yu2hy\OneDrive\◎2020_copus\가공_결과물\바른형태분석_말뭉치\최신버전\v\국립국어원 일상대화 말뭉치2020,2021_v5.csv
C:\Users\yu2hy\OneDrive\◎2020_copus\가공_결과물\바른형태분석_말뭉치\최신버전\v\세종_문어_형태분석_말뭉치_v5.csv
%%%%%1.2 문장 파일-sen2 읽기
C:\Users\yu2hy\OneDrive\◎2020_copus\가공_결과물\바른형태분석_말뭉치\최신버전\sen\국립국어원 구어 말뭉치(버전 1.2)_일부(1,15)_sen2.csv
C:\Users\yu2hy\OneDrive\◎2020_copus\가공_결과물\바른형태분석_말뭉치\최신버전\sen\국립국어원 일상대화 말뭉치2020,2021_sen2.csv
C:\Users\yu2hy\OneDrive\◎2020_copus\가공_결과물\바른형태분석_말뭉치\최신버전\sen\세종_문어_형태분석_말뭉치_sen2.csv
0, 437773
1, 434088
2, 574023


In [3]:
#%% 누락값 채움. 
#object를 category형으로 바꿈. : 45초 걸림
exclude_cols = ['form/label', 'N_form', 'V_form', 'f_V_form']  # 제외할 열
for df in df_v_list:
    for col in df.columns:
        if col in exclude_cols:
            continue  # 제외할 열은 건너뛰기

        # 숫자형이면 -1로 채움
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].fillna(-1)

        # 문자열(object)이면 'NULL'로 채우고 고유값 적을 때만 category로
        elif pd.api.types.is_object_dtype(df[col]):
            df[col] = df[col].fillna('NULL')
            unique_ratio = df[col].nunique(dropna=False) / len(df)
            if unique_ratio < 0.5:
                df[col] = df[col].astype('category')

import re
import numpy as np

# precompile regexes with NON-capturing groups (no warnings)
re_tt = re.compile(r'(?:었었|았었)')
re_t  = re.compile(r'(?:었|았)')
re_m  = re.compile(r'(?:겠|겄)')

for i, df in enumerate(df_v_list):
    # --- EP_form & flags ---
    if 'EP_form' in df.columns:
        s = df['EP_form'].astype('string').str.replace(' + ', '', regex=False)
        df['EP_form'] = s
        tt = s.str.contains(re_tt, na=False)
        df['EP_TT'] = tt
        df['EP_T']  = (~tt) & s.str.contains(re_t, na=False)
        df['EP_M']  = s.str.contains(re_m, na=False)

    # --- f_EP_form & flags ---
    if 'f_EP_form' in df.columns:
        fs = df['f_EP_form'].astype('string').str.replace(' + ', '', regex=False)
        df['f_EP_form'] = fs
        ftt = fs.str.contains(re_tt, na=False)
        df['f_EP_TT'] = ftt
        df['f_EP_T']  = (~ftt) & fs.str.contains(re_t, na=False)
        df['f_EP_M']  = fs.str.contains(re_m, na=False)

    # --- self_VX ---
    if 'vx_no.' in df.columns:
        df['self_VX'] = df['vx_no.'].ne(-1)
    else:
        # if the column is missing, default to False
        df['self_VX'] = False

#category 업데이트
# --- file_id → new category 적용 --- 
mapping_df = pd.read_csv('../label/TAG_file_Id_new_category.csv')
id_to_category = dict(zip(mapping_df['file_id'], mapping_df['category']))

for df in df_v_list:
    if 'file_id' in df.columns and 'category' in df.columns:
        df['category'] = df['file_id'].map(id_to_category).fillna(df['category'])


In [ ]:
# --
# 필터 넣어서 확인 문장 출력
# --


#필터
def filter_v(df):
    vno = pd.to_numeric(df["V_No"], errors="coerce")
    return df[df["V_No"].isin([2241, 2224, 2022, 2037])]

#df별 필터 적용 후 sen_df와 결합
out = []
for df_v, df_sen in zip(df_v_list, df_sen_list):
    merged = filter_v(df_v).merge(
        df_sen,
        on=["sen_id"],
        how="left"
    )
    out.append(merged)

#df 합치기 및 csv 파일로 저장
save_folder = Path(r"C:\Users\yu2hy\OneDrive\◎2020_copus\python_2023\bareun\csv") / "결과값"
output_file_name = save_folder / "sen" / f"merged_all_{timestamp}.csv"
print(f"Output file: {output_file_name}")
pd.concat(out, ignore_index=True).to_csv(
    output_file_name, index=False, encoding="utf-8-sig"
)


<>:22: SyntaxWarning: invalid escape sequence '\m'
<>:22: SyntaxWarning: invalid escape sequence '\m'
C:\Users\yu2hy\AppData\Local\Temp\ipykernel_21152\2272103212.py:22: SyntaxWarning: invalid escape sequence '\m'
  output_file_name = save_folder + f"\merged_all_{timestamp}.csv"


Output file: csv\결과값\sen\merged_all_2025-12-26_00-18.csv


In [4]:
df_v_list[2].head(10000)

,ID,sen_id,word_id,word_id2,form/label,N_form,N_label,V_form,V_label,EP_form,...,V_label_0,V_No,copus,EP_TT,EP_T,EP_M,f_EP_TT,f_EP_T,f_EP_M,self_VX
0,12,AA0001.7,10,10,넓히/VV + 어/EF,NaN,NULL,넓히,VV,NULL,...,VV,3743,2,False,False,False,False,False,False,False
1,14,AA0001.8,2,2,세계/NNG + 적/XSN + 이/VCP + ㄴ/ETM,세계 + 적,XSN,이,VCP,NULL,...,VP,1001,2,False,False,False,False,False,False,False
2,23,AA0001.8,11,11,나서/VV + 었/EP + 다/EF + ./SF,NaN,NULL,나서,VV,었,...,VV,3132,2,False,True,False,False,True,False,False
3,28,AA0001.9,5,5,사용하/VV + 는/ETM,NaN,NULL,사용하,VV,NULL,...,VV,3083,2,False,False,False,False,False,False,False
4,31,AA0001.9,8,8,디자인하/VV + 아/EC,NaN,NULL,디자인하,VV,NULL,...,VV,4999,2,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,42304,AA0002.1200,6,6,비하/VV + 아/EC,NaN,NULL,비하,VV,NULL,...,VV,3137,2,False,False,False,False,False,False,False
9996,42309,AA0002.1200,11,11,비하/VV + 아/EC,NaN,NULL,비하,VV,NULL,...,VV,3137,2,False,False,False,False,False,False,False
9997,42312,AA0002.1200,14,14,줄어들/VV + ㄴ/ETM,NaN,NULL,줄어들,VV,NULL,...,VV,3302,2,False,False,False,False,False,False,False
9998,42313,AA0002.1200,15,15,것/NNB + 이/VCP + 다/EF + ./SF,것,NNB,이,VCP,NULL,...,VP,1001,2,False,False,False,False,False,False,True


In [5]:
df_sen_list[1].head(10)

,ID,sen_id,form,original_form,birthplace,principal_residence,current_residence,age,sex,education,occupation,file_id,docu_num,category,공공성,상호작용성,화자수,year,S_Len
0,1,SDRW2000000001.1.1.1,저는 여행 다니는 것을,저는 여행 다니는 것을,경기,경기,경기,20대,여성,대재,학생,SDRW2000000001,1,일상대화,사적,대화,2인,2020,4
1,2,SDRW2000000001.1.1.2,굉장히 좋아하는데요.,굉장히 좋아하는데요.,경기,경기,경기,20대,여성,대재,학생,SDRW2000000001,1,일상대화,사적,대화,2인,2020,2
2,3,SDRW2000000001.1.1.3,그래가지고,그래가지고,경기,경기,경기,20대,여성,대재,학생,SDRW2000000001,1,일상대화,사적,대화,2인,2020,2
3,4,SDRW2000000001.1.1.4,스페인이나 뭐 영국,스페인이나 뭐 영국,경기,경기,경기,20대,여성,대재,학생,SDRW2000000001,1,일상대화,사적,대화,2인,2020,3
4,5,SDRW2000000001.1.1.5,유럽,유럽,경기,경기,경기,20대,여성,대재,학생,SDRW2000000001,1,일상대화,사적,대화,2인,2020,1
5,6,SDRW2000000001.1.1.6,아니면 국내에서도 뭐 강릉이나,아니면 국내에서도 뭐 강릉이나,경기,경기,경기,20대,여성,대재,학생,SDRW2000000001,1,일상대화,사적,대화,2인,2020,4
6,7,SDRW2000000001.1.1.7,전주 같은 데를 많이 다녔는데,전주 같은 데를 많이 다녔는데,경기,경기,경기,20대,여성,대재,학생,SDRW2000000001,1,일상대화,사적,대화,2인,2020,5
7,8,SDRW2000000001.1.1.8,혹시 여행 다니는 거,혹시 여행 다니는 거,경기,경기,경기,20대,여성,대재,학생,SDRW2000000001,1,일상대화,사적,대화,2인,2020,4
8,9,SDRW2000000001.1.1.9,좋아하시나요?,좋아하시나요?,경기,경기,경기,20대,여성,대재,학생,SDRW2000000001,1,일상대화,사적,대화,2인,2020,1
9,10,SDRW2000000001.1.1.10,저 여행 다니는 거 되게 좋아해서,저 여행 다니는 거 되게 좋아해서,경기,경기,경기,20대,여성,대졸,전문가 및 관련 종사자,SDRW2000000001,1,일상대화,사적,대화,2인,2020,6


In [4]:
for i, df in enumerate(df_v_list):
    print(f"{i}: {len(df)}, v: {len(df[df['V_No']!= -1])}, E, {len(df[df['EN_No']!= -1])}, v-n: {len(df[df['N_form']==None])}" )

0: 3698172, v: 3669128, E, 3690747, v-n: 0
1: 3607347, v: 3587758, E, 3596121, v-n: 0
2: 4599514, v: 4574704, E, 4597601, v-n: 0


In [ ]:
#각종 함수 선언 셀, process_dataframe, generate_pivot_statistics, generate_sentence_df, expand_window

import numpy as np
import gc
import pandas as pd
from pandas.api.types import is_categorical_dtype

### 추가 정보 붙이는 함수 선언     return result_df
import pandas as pd
import numpy as np

def process_dataframe(result_df: pd.DataFrame) -> pd.DataFrame:
    # ---- 기본 방어: Series가 들어오면 DF로 ----
    if isinstance(result_df, pd.Series):
        result_df = result_df.reset_index(name=result_df.name or "ID")

    # ---- 인덱스가 의미있는 경우만 풀기 (MultiIndex/Named index) ----
    if not isinstance(result_df.index, pd.RangeIndex) or result_df.index.name is not None:
        result_df = result_df.reset_index()

    # V_label 덧붙이기
    if ("V_label" in result_df.columns) and ("V_label_0" not in result_df.columns):
        data = {
            "V_label": ["VV", "VA", "VCN", "VCP", "XSV", "XSA", "VX"],
            "V_label_0": ["VV", "VA", "VA", "VP", "VV", "VA", "VX"]
        }
        df_v_label = pd.DataFrame(data)
        result_df = pd.merge(result_df, df_v_label, on="V_label", how="left")

    # EN 정보 덧붙이기
    if ("EN_No" in result_df.columns) and ("EN_No_0" not in result_df.columns):
        result_df["EN_No"] = pd.to_numeric(result_df["EN_No"], errors="coerce")
        result_df["EN_No_0"] = np.floor(result_df["EN_No"]).astype("Int64")  # nullable int
        result_df["EN_No_1"] = result_df["EN_No"] - result_df["EN_No_0"].astype("Float64")

    # f_EN 정보 덧붙이기
    if ("f_EN_No" in result_df.columns) and ("f_EN_No_0" not in result_df.columns):
        result_df["f_EN_No"] = pd.to_numeric(result_df["f_EN_No"], errors="coerce")
        result_df["f_EN_No_0"] = np.floor(result_df["f_EN_No"]).astype("Int64")
        result_df["f_EN_No_1"] = result_df["f_EN_No"] - result_df["f_EN_No_0"].astype("Float64")

    # EP 정보 덧붙이기
    if ("EP_form" in result_df.columns) and ("T" not in result_df.columns):
        s = result_df["EP_form"].astype("string")
        result_df["T"] = s.str.contains(r"았|었", na=False)
        result_df["M"] = s.str.contains(r"겠|겄", na=False)
        result_df["H"] = s.str.contains(r"시", na=False)

    # f_EP 정보 덧붙이기
    if ("f_EP_form" in result_df.columns) and ("f_T" not in result_df.columns):
        s = result_df["f_EP_form"].astype("string")
        result_df["f_T"] = s.str.contains(r"았|었", na=False)
        result_df["f_M"] = s.str.contains(r"겠|겄", na=False)
        result_df["f_H"] = s.str.contains(r"시", na=False)

    # V_No 정보 덧붙이기 (V_label이 없는 경우)
    if ("V_No" in result_df.columns) and ("V_label" not in result_df.columns):
        v = pd.to_numeric(result_df["V_No"], errors="coerce")

        def assign_v_label(v_no):
            if pd.isna(v_no):
                return ""
            if 0 <= v_no < 1000:
                return "VX"
            elif 1000 <= v_no < 2000:
                return "VC"
            elif 2000 <= v_no < 3000:
                return "VA"
            elif 3000 <= v_no < 5000:
                return "VV"
            else:
                return ""

        result_df["V_label"] = v.apply(assign_v_label)

    return result_df


###DataFrame에서 필터를 수행한 결과를 문장과 합해서 출력 ##return result_df_list
def generate_sentence_df(df_list, df_sen_list, filter_fn=None, window=0, process_fn=None):
    """
    DataFrame에서 필터를 수행한 결과를 문장과 합해서 출력

    Parameters:
        df_list (list of pd.DataFrame): 분석 대상 데이터프레임 리스트
        df_sen_list (list of pd.DataFrame): 합할 문장 데이터프레임 리스트
        process_fn (callable, optional): 결과 df 후처리 함수 (예: process_dataframe)
        filter_fn (callable, optional): 각 df에 필터를 적용할 함수 (예: lambda df: df[df['V_label'] == 'VX'])

    Returns:
        result_df_list: list of pd.DataFrame: 통합된 데이터프레임 리스트 결과
    """
    result_df_list = []
    print(f"*** 문장 통합 df를 만듭니다. ***")
    
    for i, (df, df_sen) in enumerate(zip(df_list, df_sen_list)):
        print(f"{i}. 통합 중...")

        # 필터 함수가 있다면 적용
        if filter_fn:
            taget_df = filter_fn(df)

            if window > 0:
                df = expand_window(taget_df, df, window)
            else: 
                df = taget_df

        df_sen = df_sen[['sen_id', 'form']]
        result_df = pd.merge(df_sen, df, how='right', on = 'sen_id')

        if process_fn:
            result_df = process_fn(result_df)
        
        print(f"{i}. 통합 완료..., 결과: {len(result_df)}")

        result_df_list.append(result_df)
    return result_df_list

###오류 후보 행들의 sen_id와 word_id2를 기준으로 주변 window만큼 확장한 행을 추출.    #return result_df
def expand_window(taget_df, df, window=2):
    """
    오류 후보 행들의 sen_id와 word_id2를 기준으로 주변 window만큼 확장한 행을 추출.
    """
    # 오류 후보들만 뽑고
    taget_row = taget_df[['sen_id', 'word_id2']].dropna().copy()
    taget_row['word_id2'] = taget_row['word_id2'].astype(int)

    # window 범위로 확장
    expanded_rows = []
    for _, row in taget_row.iterrows():
        sid = row['sen_id']
        wid = row['word_id2']
        for offset in range(-window, window + 1):
            expanded_rows.append((sid, wid + offset))

    # 중복 제거
    expanded_df = pd.DataFrame(expanded_rows, columns=['sen_id', 'word_id2']).drop_duplicates()

    # 기준 df와 merge
    df['word_id2'] = df['word_id2'].astype('Int64')  # Null 허용 정수로
    result_df = pd.merge(df, expanded_df, on=['sen_id', 'word_id2'], how='inner')
    result_df = result_df.sort_values(by=['sen_id', 'word_id2']).reset_index(drop=True)

    return result_df

#개선된 그룹바이 함수

import pandas as pd
import numpy as np
import gc

#wrapper: Series → DataFrame 으로 감싸고 process_dataframe를 대신 실행해 주고 다시 Series로 되돌려서 반환
def make_process_fn_for_series(index_columns, df_process_fn):
    """Series -> DF -> df_process_fn -> Series 래퍼"""
    def _process_fn(s: pd.Series) -> pd.Series:
        df = s.reset_index(name=s.name or "ID")
        df = df_process_fn(df)
        return df.set_index(index_columns)["ID"].rename(s.name or "ID")
    return _process_fn


def generate_pivot_statistics(
    df_list,
    index_columns,
    filter_fn=None,
    process_fn=None  # (1) Series를 받는 함수도 OK, (2) DF를 받는 함수도 OK
):
    print("*** 통계를 만듭니다. (관측 조합, NaN 포함) ***")

    total_series = None
    total_rows = 0

    # ✅ process_fn이 DF용이면 Series용으로 감싸기 (루프 밖에서 1번만)
    if process_fn is not None:
        # 간단 휴리스틱: "DF를 받는 함수"로 쓰려는 경우를 대비해 래핑 버전도 준비
        process_series_fn = make_process_fn_for_series(index_columns, process_fn)
    else:
        process_series_fn = None

    for i, df in enumerate(df_list):
        print(f"df_{i}. 시작합니다. len(df): {len(df)}")

        if filter_fn is not None:
            df = filter_fn(df)
            print(f"df_{i}. 필터 적용 후 len(df): {len(df)}")

        total_rows += len(df)

        try:
            s = (
                df.groupby(index_columns, dropna=False, observed=True, sort=False)
                  .size()
                  .rename("ID")
            )
        except TypeError:
            SENTINEL = "__<NA>__"
            sub = df[index_columns].copy()
            for c in index_columns:
                sub[c] = sub[c].where(sub[c].notna(), SENTINEL)
            s = sub.value_counts(sort=False).rename("ID")
            s.index = pd.MultiIndex.from_tuples(
                [tuple(np.nan if x == SENTINEL else x for x in tup) for tup in s.index],
                names=index_columns
            )

        # ✅ 후처리 (있을 때만)
        if process_series_fn is not None:
            s = process_series_fn(s)

        total_series = s if total_series is None else total_series.add(s, fill_value=0)

        del s, df
        gc.collect()

    return total_series


# Null값 등으로 인해서 제외되는 값이 있는지 확인하는 함수
def debug_drop_reason(df, index_columns):
    n_all = len(df)
    n_key_na = df[index_columns].isna().any(axis=1).sum()     # 키에 NA 있는 행 수
    n_id_na = df['ID'].isna().sum() if 'ID' in df.columns else 0

    # pivot_table식 count가 실제로 세는 개수(대략적인 감산식)
    approx_pivot_count = df.loc[~df[index_columns].isna().any(axis=1), 'ID'].notna().sum()

    # 권장 경로: groupby + size (키 NA 포함)
    gb_size = df.groupby(index_columns, dropna=False, observed=True).size().sum()

    return {
        "len(df)": int(n_all),
        "키에 NA 있는 행": int(n_key_na),
        "ID가 NA인 행": int(n_id_na),
        "pivot_table(count) 합": int(approx_pivot_count),
        "groupby(size, dropna=False) 합": int(gb_size),
    }

In [ ]:
### 전체 동사 및 어미 수
prefix = "V_E"
index_columns = ['V_No', 'vx_len', 'vx_no.', 'EP_TT','EP_T','EP_M', 'EN_No', 'EN_label', 'vx1_No', 'f_EP_TT','f_EP_T','f_EP_M', 'f_EN_No', 'f_EN_label','copus', 'category']

#에러 확인
for i, df in(enumerate(df_v_list)):
    print(debug_drop_reason(df, index_columns))

result_df = generate_pivot_statistics(
    df_list=df_v_list,
    index_columns=index_columns,
    process_fn=process_dataframe,
    #filter_fn=filter
)
print("result_df length:", len(result_df))

# 결과 DataFrame 저장
result_df_total = result_df.astype('int32').to_frame().reset_index()
save_folder = Path(r"C:\Users\yu2hy\OneDrive\◎2020_copus\python_2023\bareun\csv") 
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
output_file_name = save_folder / "통계용" / f"V_VX_Category_{timestamp}.csv"
output_file_name.parent.mkdir(parents=True, exist_ok=True)  # 폴더 생성
result_df_total.to_csv(output_file_name, index=True, encoding='utf-8-sig')
print(f"*** 저장 완료: {output_file_name} ({len(result_df_total):,}행) ***")

{'len(df)': 3698172, '키에 NA 있는 행': 0, 'ID가 NA인 행': 0, 'pivot_table(count) 합': 3698172, 'groupby(size, dropna=False) 합': 3698172}
{'len(df)': 3607347, '키에 NA 있는 행': 0, 'ID가 NA인 행': 0, 'pivot_table(count) 합': 3607347, 'groupby(size, dropna=False) 합': 3607347}
{'len(df)': 4599514, '키에 NA 있는 행': 0, 'ID가 NA인 행': 0, 'pivot_table(count) 합': 4599514, 'groupby(size, dropna=False) 합': 4599514}
*** 통계를 만듭니다. (관측 조합, NaN 포함) ***
df_0. 시작합니다. len(df): 3698172
df_1. 시작합니다. len(df): 3607347
df_2. 시작합니다. len(df): 4599514


In [ ]:
#결과가 100만 행 넘으면 쪼개서 다시 저장
if len(result_df_total['ID']) >1000000:
    print("경고: 결과 합계가 100만 행을 넘습니다. 따로따로 저장합니다.")
    for i in range(0, len(result_df_total), 1000000):
        chunk = result_df_total.iloc[i:i+1000000]
        timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
        print(f"***저장합니다.:    {datetime.now()}***")
        out_file = dir_path / save_folder / f'{prefix}_statistics_{timestamp}_{i//1000000}.csv'
        chunk.to_csv(out_file, index=True, encoding='utf-8-sig')
        print(f"*** 저장 완료: {out_file} ({len(chunk):,}행) ***")

***저장합니다.:    2025-12-26 00:47:37.323884***
*** 저장 완료: C:\Users\yu2hy\OneDrive\◎2020_copus\python_2023\bareun\csv\결과값\V_E_statistics_2025-12-26_00-47.csv (1,331,413행) ***
경고: 결과 합계가 100만 행을 넘습니다. 따로따로 저장합니다.
***저장합니다.:    2025-12-26 00:47:41.800752***
*** 저장 완료: C:\Users\yu2hy\OneDrive\◎2020_copus\python_2023\bareun\csv\결과값\V_E_statistics_2025-12-26_00-47_0.csv (1,000,000행) ***
***저장합니다.:    2025-12-26 00:47:45.181174***
*** 저장 완료: C:\Users\yu2hy\OneDrive\◎2020_copus\python_2023\bareun\csv\결과값\V_E_statistics_2025-12-26_00-47_1.csv (331,413행) ***


In [ ]:

# 결과 DataFrame
result_df_total = result_df.astype('int32').to_frame().reset_index()
if len(result_df_total['ID']) >1000000:
    print("경고: 결과 합계가 100만 행을 넘습니다. 따로따로 저장합니다.")
    for i in range(0, len(result_df_total), 1000000):
        chunk = result_df_total.iloc[i:i+1000000]
        timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
        print(f"***저장합니다.:    {datetime.now()}***")
        out_file = dir_path / save_folder / f'{prefix}_statistics_{timestamp}_{i//1000000}.csv'
        chunk.to_csv(out_file, index=True, encoding='utf-8-sig')
        print(f"*** 저장 완료: {out_file} ({len(chunk):,}행) ***")
else:
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
    print(f"***저장합니다.:    {datetime.now()}***")
    out_file = dir_path / save_folder / f'{prefix}_statistics_{timestamp}.csv'
    result_df_total.to_csv(out_file, index=True, encoding='utf-8-sig')
    print(f"*** 저장 완료: {out_file} ({len(result_df_total):,}행) ***")


In [16]:
### file_id, 동사, -었-, 결합, vx를 나누어서 확인. 전체를 함께 하면 600만라인을 넘어감.
prefix = "V_VX_E"

# file_id, 동사, -었-, 결합
tag1 = "file동사-었-결합"
index_columns_1 = ['V_No', 'vx_no.', 'vx_len', 'self_VX', 'EP_TT','EP_T','EP_M', 'EN_No', 'EN_label', 'copus', 'file_id'] # file_id, 동사, -었-, 결합

tag2 = "category-었-결합"
index_columns_2 = ['V_No', 'vx_len', 'vx_no.', 'EP_TT','EP_T','EP_M', 'EN_No', 'EN_label', 'copus', 'category']

tag3 = "category-었-결합_vx1"
index_columns_3 = ['V_No', 'vx_len', 'EP_TT','EP_T','EP_M', 'EN_No', 'EN_label', 'vx1_No','f_EP_TT','f_EP_T','f_EP_M', 'f_EN_No', 'f_EN_label','copus', 'category']
filter3 = lambda df: df[df['vx_len'] > 0]

#에러 확인
for i, df in(enumerate(df_v_list)):
    print(debug_drop_reason(df, index_columns_2))

result_df = generate_pivot_statistics(
    df_list=df_v_list,
    index_columns=index_columns_2,
    process_fn=process_dataframe,
    #filter_fn=filter3
)

# 결과 DataFrame
result_df_total = result_df.astype('int32').to_frame().reset_index()
if len(result_df_total['ID']) >1000000:
    print("경고: 결과 합계가 100만 행을 넘습니다. 따로따로 저장합니다.")
    for i in range(0, len(result_df_total), 1000000):
        chunk = result_df_total.iloc[i:i+1000000]
        timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
        print(f"***저장합니다.:    {datetime.now()}***")
        out_file = dir_path / save_folder / f'{prefix}_{tag2}_{timestamp}_{i//1000000}.csv'
        chunk.to_csv(out_file, index=True, encoding='utf-8-sig')
        print(f"*** 저장 완료: {out_file} ({len(chunk):,}행) ***")
else:
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
    print(f"***저장합니다.:    {datetime.now()}***")
    out_file = dir_path / save_folder / f'{prefix}_{tag2}_{timestamp}.csv'
    result_df_total.to_csv(out_file, index=True, encoding='utf-8-sig')
    print(f"*** 저장 완료: {out_file} ({len(result_df_total):,}행) ***")



{'len(df)': 3698172, '키에 NA 있는 행': 0, 'ID가 NA인 행': 0, 'pivot_table(count) 합': 3698172, 'groupby(size, dropna=False) 합': 3698172}
{'len(df)': 3607347, '키에 NA 있는 행': 0, 'ID가 NA인 행': 0, 'pivot_table(count) 합': 3607347, 'groupby(size, dropna=False) 합': 3607347}
{'len(df)': 4599514, '키에 NA 있는 행': 0, 'ID가 NA인 행': 0, 'pivot_table(count) 합': 4599514, 'groupby(size, dropna=False) 합': 4599514}
*** 통계를 만듭니다. (관측 조합, NaN 포함) ***
df_0. 시작합니다. len(df): 3698172
df_1. 시작합니다. len(df): 3607347
df_2. 시작합니다. len(df): 4599514
***저장합니다.:    2025-12-26 00:31:39.064940***


OSError: Cannot save file into a non-existent directory: 'C:\Users\yu2hy\OneDrive\◎2020_copus\가공_결과물\바른형태분석_말뭉치\csv\결과값\sen'

In [10]:
df_v_list[0]

,ID,sen_id,word_id,word_id2,form/label,N_form,N_label,V_form,V_label,EP_form,...,V_label_0,V_No,copus,EP_TT,EP_T,EP_M,f_EP_TT,f_EP_T,f_EP_M,self_VX
0,4,SBRW1900012479.1.1.3,2,2,안녕하/VA + 시/EP + ㅂ니까/EF + ?/SF,NaN,NULL,안녕하,VA,시,...,VA,2058,0,False,False,False,False,False,False,False
1,11,SBRW1900012479.1.1.4,7,7,비화되/VV + 곤/EC,NaN,NULL,비화되,VV,NULL,...,VV,4999,0,False,False,False,False,False,False,False
2,12,SBRW1900012479.1.1.4,8,8,하/VX + ㅂ니다/EF + ./SF,NaN,NULL,하,VX,NULL,...,VX,102,0,False,False,False,False,False,False,True
3,15,SBRW1900012479.1.1.5,3,3,밝혀지/VV + ㄴ/ETM,NaN,NULL,밝혀지,VV,NULL,...,VV,3388,0,False,False,False,False,False,False,False
4,21,SBRW1900012479.1.1.5,9,9,불러오/VV + ㄴ/ETM,NaN,NULL,불러오,VV,NULL,...,VV,4999,0,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3698167,5449162,SBRW1900000475.1.1.774,3,3,없/VA + 고/EC,NaN,NULL,없,VA,NULL,...,VA,2004,0,False,False,False,False,False,False,False
3698168,5449172,SBRW1900000475.1.1.774,13,13,지원금/NNG + 이/VCP + 에요/EF + ./SF,지원금,NNG,이,VCP,NULL,...,VP,1001,0,False,False,False,False,False,False,False
3698169,5449174,SBRW1900000475.1.1.775,2,2,필요하/VA + ㄴ데/EC,NaN,NULL,필요하,VA,NULL,...,VA,2015,0,False,False,False,False,False,False,False
3698170,5449175,SBRW1900000475.1.1.775,3,3,공개하/VV + 자/EF + ./SF,NaN,NULL,공개하,VV,NULL,...,VV,3480,0,False,False,False,False,False,False,False


In [13]:
### vx부류와 결합한 문장 출력
prefix = "VX"

def filter_vx(df: pd.DataFrame) -> pd.DataFrame:
    vx_list = [101, 102, 111, 112, 113, 114, 122, 123, 124, 125, 126, 131, 132]
    vx_cols = ['vx1_No', 'vx2_No', 'vx3_No', 'vx4_No', 'vx5_No', 'vx_no.']

    # keep only columns that actually exist
    cols = [c for c in vx_cols if c in df.columns]
    if not cols:
        return df.iloc[0:0]  # empty

    # ensure numeric comparison & build a row-wise "any" mask
    mask = pd.DataFrame({
        c: pd.to_numeric(df[c], errors='coerce').isin(vx_list)
        for c in cols
    }).any(axis=1)

    return df[mask.fillna(False)]

result_df_list = generate_sentence_df(
    df_list=df_v_list,
    df_sen_list=df_sen_list,
    process_fn=process_dataframe,
    filter_fn=filter_vx,
    window=0
)
## 문장 파일 각각 저장하기.
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
print(f"***저장합니다.:    {datetime.now()}***")

for i, result_df in enumerate(result_df_list):
    out_file = dir_path / save_folder / f'{prefix}_sentence_{i}_{timestamp}.csv'
    result_df.to_csv(out_file, index=True, encoding='utf-8-sig')
    print(f"*** 저장 완료: {out_file} ({len(result_df):,}행) ***")

*** 문장 통합 df를 만듭니다. ***
0. 통합 중...
0. 통합 완료..., 결과: 232924
1. 통합 중...
1. 통합 완료..., 결과: 158578
2. 통합 중...
2. 통합 완료..., 결과: 481654
***저장합니다.:    2025-09-19 17:38:19.105565***
*** 저장 완료: C:\Users\yu2hy\OneDrive\◎2020_copus\가공_결과물\바른형태분석_말뭉치\2025.09.01_상\VX_sentence_0_2025-09-19_17-38.csv (232,924행) ***
*** 저장 완료: C:\Users\yu2hy\OneDrive\◎2020_copus\가공_결과물\바른형태분석_말뭉치\2025.09.01_상\VX_sentence_1_2025-09-19_17-38.csv (158,578행) ***
*** 저장 완료: C:\Users\yu2hy\OneDrive\◎2020_copus\가공_결과물\바른형태분석_말뭉치\2025.09.01_상\VX_sentence_2_2025-09-19_17-38.csv (481,654행) ***


In [ ]:
##문장 파일 통합해서 저장하기
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
result_df = pd.concat(result_df_list)
out_file = dir_path / save_folder / f'{prefix}_sentence_{i}_{timestamp}.csv'
result_df.to_csv(out_file, index=True, encoding='utf-8-sig')